# Object Detection：SSD + MobileNetV3 で物体検出する

このノートブックでは，画像分類から一歩進んで，画像内の「どこに」「何が」あるかを推論する物体検出（Object Detection）を扱う。

最初に，物体検出，特徴抽出機，SSD，MobileNetV3 の役割を整理する。その後，TorchVision の `ssdlite320_mobilenet_v3_large` を使い，SSD + MobileNetV3 で物体検出を行う。最後に，カメラで撮影したグーチョキパー動画を教師データ化し，独自データでファインチューニングする流れを確認する。


<style>
.od-figure {
  border: 1px solid #d9e2ec;
  border-radius: 8px;
  background: #f8fbff;
  padding: 14px;
  margin: 14px 0 20px;
  overflow-x: auto;
}
.od-figure svg {
  max-width: 100%;
  height: auto;
  display: block;
}
.od-label {
  font: 600 14px system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  fill: #102a43;
}
.od-small {
  font: 12px system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  fill: #334e68;
}
.od-note {
  margin-top: 8px;
  color: #334e68;
  font-size: 0.92rem;
}
.od-box { fill: #ffffff; stroke: #486581; stroke-width: 1.5; rx: 8; }
.od-blue { fill: #e0f2fe; stroke: #0284c7; }
.od-green { fill: #dcfce7; stroke: #16a34a; }
.od-amber { fill: #fef3c7; stroke: #d97706; }
.od-red { fill: #fee2e2; stroke: #dc2626; }
.od-purple { fill: #ede9fe; stroke: #7c3aed; }
.od-arrow { stroke: #486581; stroke-width: 2; fill: none; marker-end: url(#arrow); }
.od-dash { stroke: #64748b; stroke-width: 2; stroke-dasharray: 6 4; fill: none; }
@keyframes od-scan { 0% { transform: translateX(0); opacity: .15; } 45% { opacity: .85; } 100% { transform: translateX(260px); opacity: .15; } }
@keyframes od-pulse { 0%, 100% { opacity: .25; } 50% { opacity: 1; } }
@keyframes od-shift { 0%, 100% { transform: translate(0, 0); } 50% { transform: translate(18px, -8px); } }
.od-scanbar { animation: od-scan 3.2s ease-in-out infinite; transform-box: fill-box; }
.od-pulse { animation: od-pulse 2.2s ease-in-out infinite; }
.od-shift { animation: od-shift 2.8s ease-in-out infinite; transform-box: fill-box; }
</style>
<div class="od-figure">
<svg viewBox="0 0 900 260" role="img" aria-label="SSD + MobileNetV3 の全体像">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="30" y="60" width="150" height="120" class="od-box od-blue"/><text x="105" y="112" text-anchor="middle" class="od-label">入力画像</text><text x="105" y="138" text-anchor="middle" class="od-small">RGB image</text>
<path d="M190 120 H260" class="od-arrow"/>
<rect x="270" y="45" width="190" height="150" class="od-box od-green"/><text x="365" y="100" text-anchor="middle" class="od-label">MobileNetV3</text><text x="365" y="126" text-anchor="middle" class="od-small">特徴マップを作る</text><text x="365" y="150" text-anchor="middle" class="od-small">backbone</text>
<path d="M470 120 H540" class="od-arrow"/>
<rect x="550" y="45" width="160" height="150" class="od-box od-amber"/><text x="630" y="98" text-anchor="middle" class="od-label">SSD head</text><text x="630" y="124" text-anchor="middle" class="od-small">クラス</text><text x="630" y="148" text-anchor="middle" class="od-small">ボックス補正</text>
<path d="M720 120 H790" class="od-arrow"/>
<rect x="800" y="60" width="70" height="120" class="od-box od-red"/><text x="835" y="112" text-anchor="middle" class="od-label">検出</text><text x="835" y="138" text-anchor="middle" class="od-small">label + box</text>
</svg>

<div class="od-note">このノートでは，画像を特徴マップへ変換する部分と，物体の種類・位置を予測する部分を分けて考える。</div>
</div>


## このノート全体の流れ

1. 画像分類と物体検出の違いを確認する。
2. 「MobileNetV3 を特徴抽出機として使う」とは何かを確認する。
3. SSD が何を予測しているのかを確認する。
4. COCO で事前学習済みの SSD + MobileNetV3 で物体検出を試す。
5. グーチョキパーの教師データを動画から作る準備をする。
6. バウンディングボックス対応のデータ拡張を使う。
7. SSD + MobileNetV3 をグーチョキパー検出用にファインチューニングする。

## 到達目標

- 画像分類と物体検出の出力の違いを説明できる。
- MobileNetV3 が特徴抽出機としてどの部分を担当するかを説明できる。
- SSD が既定の候補領域に対して，クラスと位置のずれを予測するモデルであることを説明できる。
- `torchvision` の物体検出モデルに渡す画像と教師データの形式を理解できる。
- グーチョキパー動画からフレームを切り出し，アノテーションして，ファインチューニング用データにする流れを説明できる。


<div class="od-figure">
<svg viewBox="0 0 920 220" role="img" aria-label="ノート全体の流れ">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="20" y="60" width="120" height="80" class="od-box od-blue"/><text x="80" y="94" text-anchor="middle" class="od-label">概念</text><text x="80" y="118" text-anchor="middle" class="od-small">検出とは</text>
<path d="M150 100 H215" class="od-arrow"/>
<rect x="225" y="60" width="135" height="80" class="od-box od-green"/><text x="292" y="94" text-anchor="middle" class="od-label">Backbone</text><text x="292" y="118" text-anchor="middle" class="od-small">MobileNetV3</text>
<path d="M370 100 H435" class="od-arrow"/>
<rect x="445" y="60" width="120" height="80" class="od-box od-amber"/><text x="505" y="94" text-anchor="middle" class="od-label">SSD</text><text x="505" y="118" text-anchor="middle" class="od-small">候補ボックス</text>
<path d="M575 100 H640" class="od-arrow"/>
<rect x="650" y="60" width="120" height="80" class="od-box od-purple"/><text x="710" y="94" text-anchor="middle" class="od-label">推論</text><text x="710" y="118" text-anchor="middle" class="od-small">COCO</text>
<path d="M780 100 H845" class="od-arrow"/>
<rect x="855" y="60" width="50" height="80" class="od-box od-red"/><text x="880" y="94" text-anchor="middle" class="od-label">実践</text><text x="880" y="118" text-anchor="middle" class="od-small">RPS</text>
</svg>

<div class="od-note">左から順に読むと，理論から実データでのファインチューニングへ進む。</div>
</div>


## 準備

初回実行時は，COCO で事前学習済みの SSD + MobileNetV3 の重みと，MobileNetV3 の ImageNet 事前学習済み重みをダウンロードする。インターネット接続が必要である。

物体検出モデルは，訓練時には画像のリストと教師データのリストを受け取り，損失の辞書を返す。推論時には，同じ画像のリストを受け取り，検出結果のリストを返す。この動きは通常の画像分類モデルと異なる。


<div class="od-figure">
<svg viewBox="0 0 820 230" role="img" aria-label="訓練時と推論時の入出力">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="35" y="35" width="180" height="70" class="od-box od-blue"/><text x="125" y="66" text-anchor="middle" class="od-label">訓練時</text><text x="125" y="88" text-anchor="middle" class="od-small">images + targets</text>
<path d="M225 70 H325" class="od-arrow"/>
<rect x="335" y="35" width="160" height="70" class="od-box od-green"/><text x="415" y="66" text-anchor="middle" class="od-label">model.train()</text><text x="415" y="88" text-anchor="middle" class="od-small">損失を返す</text>
<path d="M505 70 H605" class="od-arrow"/>
<rect x="615" y="35" width="165" height="70" class="od-box od-amber"/><text x="697" y="66" text-anchor="middle" class="od-label">loss dict</text><text x="697" y="88" text-anchor="middle" class="od-small">分類 + box</text>
<rect x="35" y="135" width="180" height="70" class="od-box od-blue"/><text x="125" y="166" text-anchor="middle" class="od-label">推論時</text><text x="125" y="188" text-anchor="middle" class="od-small">images only</text>
<path d="M225 170 H325" class="od-arrow"/>
<rect x="335" y="135" width="160" height="70" class="od-box od-green"/><text x="415" y="166" text-anchor="middle" class="od-label">model.eval()</text><text x="415" y="188" text-anchor="middle" class="od-small">予測を返す</text>
<path d="M505 170 H605" class="od-arrow"/>
<rect x="615" y="135" width="165" height="70" class="od-box od-red"/><text x="697" y="166" text-anchor="middle" class="od-label">boxes labels scores</text><text x="697" y="188" text-anchor="middle" class="od-small">検出結果</text>
</svg>

<div class="od-note">TorchVision の検出モデルは，訓練時と推論時で返す値が変わる。</div>
</div>


In [ ]:
from __future__ import annotations

import random
from collections import defaultdict
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import tv_tensors
from torchvision.models import MobileNet_V3_Large_Weights
from torchvision.models.detection import (
    SSDLite320_MobileNet_V3_Large_Weights,
    ssdlite320_mobilenet_v3_large,
)
from torchvision.transforms import v2 as transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("../data")
RPS_ROOT = DATA_ROOT / "rps_detection"

print(f"device: {DEVICE}")


## 物体検出とは何か

画像分類では，1枚の画像全体に対して1つのラベルを予測する。たとえば，画像全体を見て「これは犬である」と分類する。

物体検出では，画像内にある物体ごとに，次の2つを同時に予測する。

- `class`：その物体が何であるか
- `box`：その物体が画像のどこにあるか

`box` はバウンディングボックスと呼ばれ，通常は左上と右下の座標 `(xmin, ymin, xmax, ymax)` で表す。したがって，物体検出の出力は「犬が画像のこの矩形にある」「人が画像のこの矩形にある」のような複数の予測になる。


<div class="od-figure">
<svg viewBox="0 0 820 300" role="img" aria-label="画像分類と物体検出の違い">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="40" width="260" height="180" class="od-box od-blue"/><circle cx="130" cy="120" r="34" fill="#93c5fd"/><rect x="188" y="84" width="52" height="80" fill="#bfdbfe" stroke="#0284c7"/><text x="170" y="252" text-anchor="middle" class="od-label">画像分類</text><text x="170" y="274" text-anchor="middle" class="od-small">画像全体 → cat</text>
<path d="M330 130 H455" class="od-arrow"/>
<rect x="490" y="40" width="260" height="180" class="od-box od-green"/><circle cx="580" cy="120" r="34" fill="#bbf7d0"/><rect x="638" y="84" width="52" height="80" fill="#dcfce7" stroke="#16a34a"/><rect x="540" y="82" width="80" height="78" fill="none" stroke="#dc2626" stroke-width="4"/><rect x="628" y="74" width="76" height="102" fill="none" stroke="#d97706" stroke-width="4"/><text x="620" y="252" text-anchor="middle" class="od-label">物体検出</text><text x="620" y="274" text-anchor="middle" class="od-small">物体ごと → label + box</text>
</svg>

<div class="od-note">分類は1枚の画像に1つの答えを出す。検出は画像内の複数物体について，種類と位置を出す。</div>
</div>


## MobileNetV3 を特徴抽出機として使うとは

CNN は，入力画像をすぐにクラス名へ変換しているわけではない。前半の畳み込み層では，エッジ，模様，部品のような特徴を段階的に取り出している。この前半部分を特徴抽出機（feature extractor, backbone）と呼ぶ。

MobileNetV3 は，軽量で高速な CNN である。物体検出では，MobileNetV3 を画像から特徴マップを作る部品として使い，その特徴マップの上に検出用の層を追加する。

つまり，ここでの MobileNetV3 の役割は「画像を分類すること」ではなく，「検出しやすい特徴マップを作ること」である。


<div class="od-figure">
<svg viewBox="0 0 900 260" role="img" aria-label="MobileNetV3 が特徴マップを作る図">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="35" y="70" width="130" height="100" class="od-box od-blue"/><text x="100" y="116" text-anchor="middle" class="od-label">画像</text><text x="100" y="140" text-anchor="middle" class="od-small">320 x 320</text>
<path d="M175 120 H245" class="od-arrow"/>
<g class="od-shift"><rect x="260" y="50" width="140" height="140" class="od-box od-green"/><line x1="285" y1="80" x2="375" y2="80" stroke="#16a34a"/><line x1="285" y1="120" x2="375" y2="120" stroke="#16a34a"/><line x1="285" y1="160" x2="375" y2="160" stroke="#16a34a"/><text x="330" y="220" text-anchor="middle" class="od-label">畳み込み</text></g>
<path d="M420 120 H490" class="od-arrow"/>
<rect x="505" y="35" width="110" height="160" class="od-box od-amber"/><rect x="535" y="65" width="50" height="50" fill="#fde68a" stroke="#d97706"/><rect x="548" y="130" width="24" height="24" fill="#fbbf24" stroke="#d97706"/><text x="560" y="220" text-anchor="middle" class="od-label">特徴マップ</text>
<path d="M630 120 H700" class="od-arrow"/>
<rect x="715" y="55" width="145" height="130" class="od-box od-purple"/><text x="787" y="107" text-anchor="middle" class="od-label">検出ヘッド</text><text x="787" y="132" text-anchor="middle" class="od-small">分類 + 位置補正</text>
</svg>

<div class="od-note">MobileNetV3 は分類名を直接出すのではなく，検出ヘッドが使う特徴マップを作る。</div>
</div>


## これまでの MobileNetV3 だけのモデルとの違い

画像分類の MobileNetV3 では，最後に1枚の画像を1つのクラスへ分類する。

一方，SSD + MobileNetV3 では，MobileNetV3 が作った複数サイズの特徴マップに対して，検出ヘッドが多くの候補ボックスを調べる。各候補ボックスについて，次の2種類の値を予測する。

- その候補に各クラスの物体が含まれる確率
- 候補ボックスをどのくらい移動・拡大縮小すれば物体に合うか

そのため，画像分類モデルよりも出力が複雑になる。分類モデルの出力は `num_classes` 個の logits だが，物体検出モデルの出力は，多数のボックス，ラベル，信頼度スコアである。


<div class="od-figure">
<svg viewBox="0 0 880 300" role="img" aria-label="MobileNetV3分類とSSD検出の比較">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="45" width="170" height="70" class="od-box od-blue"/><text x="125" y="86" text-anchor="middle" class="od-label">MobileNetV3</text>
<path d="M220 80 H310" class="od-arrow"/><rect x="320" y="45" width="150" height="70" class="od-box od-green"/><text x="395" y="75" text-anchor="middle" class="od-label">分類ヘッド</text><text x="395" y="98" text-anchor="middle" class="od-small">1つのラベル</text>
<rect x="40" y="175" width="170" height="70" class="od-box od-blue"/><text x="125" y="216" text-anchor="middle" class="od-label">MobileNetV3</text>
<path d="M220 210 H310" class="od-arrow"/><rect x="320" y="175" width="150" height="70" class="od-box od-amber"/><text x="395" y="205" text-anchor="middle" class="od-label">SSD head</text><text x="395" y="228" text-anchor="middle" class="od-small">多くの候補</text>
<path d="M480 210 H570" class="od-arrow"/><rect x="580" y="155" width="250" height="110" class="od-box od-red"/><text x="705" y="196" text-anchor="middle" class="od-label">boxes + labels + scores</text><text x="705" y="224" text-anchor="middle" class="od-small">複数の物体を返す</text>
</svg>

<div class="od-note">同じ MobileNetV3 でも，最後に接続するヘッドが変わるとタスクと出力形式が変わる。</div>
</div>


## SSD とは

SSD は Single Shot MultiBox Detector の略である。Single Shot とは，画像を1回モデルに通すだけで検出結果を出すという意味である。

SSD は画像上に多数の既定ボックス（default boxes, anchor boxes）を用意し，各ボックスに対して次を予測する。

- どのクラスの物体らしいか
- 既定ボックスからどれだけ位置をずらすか

同じ物体に対して似たボックスが何個も出るため，最後に NMS（Non-Maximum Suppression）で重なりすぎた候補を減らす。これにより，信頼度が高く，重複の少ない検出結果だけを残す。


<div class="od-figure">
<svg viewBox="0 0 860 300" role="img" aria-label="SSD の候補ボックスと補正">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="50" y="40" width="250" height="200" class="od-box od-blue"/>
<g class="od-pulse"><rect x="85" y="75" width="70" height="70" fill="none" stroke="#0284c7" stroke-width="2"/><rect x="170" y="70" width="95" height="95" fill="none" stroke="#0284c7" stroke-width="2"/><rect x="110" y="150" width="130" height="60" fill="none" stroke="#0284c7" stroke-width="2"/></g>
<text x="175" y="270" text-anchor="middle" class="od-label">多数の既定ボックス</text>
<path d="M320 140 H430" class="od-arrow"/>
<rect x="450" y="55" width="170" height="170" class="od-box od-amber"/><text x="535" y="110" text-anchor="middle" class="od-label">各候補で予測</text><text x="535" y="140" text-anchor="middle" class="od-small">class score</text><text x="535" y="165" text-anchor="middle" class="od-small">box offset</text>
<path d="M640 140 H730" class="od-arrow"/>
<rect x="745" y="75" width="70" height="90" fill="none" stroke="#dc2626" stroke-width="5"/><text x="780" y="210" text-anchor="middle" class="od-label">NMS後</text>
</svg>

<div class="od-note">SSD は候補ボックスを多数並べ，クラスと位置補正を一度に予測する。</div>
</div>


## COCO で事前学習済みの SSD + MobileNetV3 を読み込む

まずはファインチューニングを行わず，COCO データセットで事前学習済みのモデルを使って推論する。COCO には人，車，犬，猫などの日常物体が含まれる。

ここで使う `ssdlite320_mobilenet_v3_large` は，MobileNetV3 を backbone として使う軽量な SSD 系モデルである。`SSDLite` は，通常の畳み込みより軽い depthwise separable convolution を使うことで，計算量を抑えた SSD である。


<div class="od-figure">
<svg viewBox="0 0 820 230" role="img" aria-label="事前学習済みモデルの読み込み">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="65" width="180" height="90" class="od-box od-purple"/><text x="130" y="103" text-anchor="middle" class="od-label">COCO weights</text><text x="130" y="128" text-anchor="middle" class="od-small">一般物体で学習済み</text>
<path d="M235 110 H330" class="od-arrow"/>
<rect x="345" y="45" width="190" height="130" class="od-box od-green"/><text x="440" y="92" text-anchor="middle" class="od-label">SSDLite320</text><text x="440" y="118" text-anchor="middle" class="od-small">MobileNetV3 backbone</text><text x="440" y="144" text-anchor="middle" class="od-small">SSD head</text>
<path d="M550 110 H645" class="od-arrow"/>
<rect x="660" y="65" width="125" height="90" class="od-box od-red"/><text x="722" y="103" text-anchor="middle" class="od-label">推論準備</text><text x="722" y="128" text-anchor="middle" class="od-small">eval mode</text>
</svg>

<div class="od-note">重み・前処理・カテゴリ名をセットで扱うと，事前学習時と同じ条件で推論できる。</div>
</div>


In [ ]:
weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
coco_detector = ssdlite320_mobilenet_v3_large(weights=weights)
coco_detector = coco_detector.to(DEVICE)
coco_detector.eval()

preprocess = weights.transforms()
coco_categories = weights.meta["categories"]

print(coco_detector.__class__.__name__)
print(f"COCO classes: {len(coco_categories)}")


## 画像を読み込んで推論する

`weights.transforms()` は，事前学習時と同じ前処理を行うための変換である。物体検出モデルでは，画像を `Tensor` に変換し，画像のリストとしてモデルへ渡す。


<div class="od-figure">
<svg viewBox="0 0 840 240" role="img" aria-label="推論の前処理とモデル入力">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="35" y="70" width="140" height="90" class="od-box od-blue"/><text x="105" y="108" text-anchor="middle" class="od-label">PIL Image</text><text x="105" y="132" text-anchor="middle" class="od-small">読み込み</text>
<path d="M185 115 H275" class="od-arrow"/>
<rect x="290" y="70" width="160" height="90" class="od-box od-green"/><text x="370" y="108" text-anchor="middle" class="od-label">preprocess</text><text x="370" y="132" text-anchor="middle" class="od-small">Tensor化</text>
<path d="M460 115 H550" class="od-arrow"/>
<rect x="565" y="70" width="110" height="90" class="od-box od-amber"/><text x="620" y="108" text-anchor="middle" class="od-label">[image]</text><text x="620" y="132" text-anchor="middle" class="od-small">リスト</text>
<path d="M685 115 H775" class="od-arrow"/>
<rect x="790" y="70" width="35" height="90" class="od-box od-red"/><text x="807" y="108" text-anchor="middle" class="od-label">出力</text>
</svg>

<div class="od-note">検出モデルには，1枚だけでも画像をリストにして渡す。</div>
</div>


In [ ]:
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(urlopen(image_url)).convert("RGB")
image


In [ ]:
input_tensor = preprocess(image).to(DEVICE)

with torch.inference_mode():
    predictions = coco_detector([input_tensor])

prediction = {key: value.cpu() for key, value in predictions[0].items()}
print(prediction.keys())
print(prediction["boxes"].shape, prediction["labels"].shape, prediction["scores"].shape)


## 検出結果を描画する

推論結果には，`boxes`，`labels`，`scores` が含まれる。信頼度が低い候補まで描くと見づらくなるため，ここではスコアがしきい値以上のものだけを描画する。


<div class="od-figure">
<svg viewBox="0 0 760 260" role="img" aria-label="検出結果の描画">
<rect x="55" y="35" width="260" height="180" class="od-box od-blue"/>
<rect x="115" y="75" width="90" height="95" fill="none" stroke="#dc2626" stroke-width="5"/>
<rect x="115" y="52" width="95" height="22" fill="#dc2626"/><text x="162" y="68" text-anchor="middle" fill="white" font-size="13" font-family="system-ui">cat: 0.93</text>
<rect x="405" y="55" width="290" height="140" class="od-box od-amber"/><text x="550" y="95" text-anchor="middle" class="od-label">boxes</text><text x="550" y="123" text-anchor="middle" class="od-label">labels</text><text x="550" y="151" text-anchor="middle" class="od-label">scores</text>
<path d="M360 125 H405" class="od-arrow"/>
</svg>

<div class="od-note">数値の出力を画像上に重ねると，モデルがどこを検出したかを確認しやすい。</div>
</div>


In [ ]:
def draw_torchvision_detections(
    image: Image.Image,
    prediction: dict[str, torch.Tensor],
    categories: list[str],
    score_threshold: float = 0.5,
) -> Image.Image:
    """TorchVision の検出結果を PIL 画像へ描画する."""

    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)

    try:
        font = ImageFont.truetype("Arial.ttf", 16)
    except OSError:
        font = ImageFont.load_default()

    for box, label_id, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"], strict=True):
        if float(score) < score_threshold:
            continue
        xyxy = [float(value) for value in box]
        label = categories[int(label_id)]
        caption = f"{label}: {float(score):.2f}"
        draw.rectangle(xyxy, outline="red", width=4)
        draw.text((xyxy[0], max(0, xyxy[1] - 18)), caption, fill="red", font=font)

    return canvas


draw_torchvision_detections(image, prediction, coco_categories, score_threshold=0.5)


## グーチョキパー検出へファインチューニングする

次に，検出対象を `gu`，`choki`，`pa` の3クラスへ変える。物体検出では背景クラスも必要なので，クラス数は背景を含めて4になる。

COCO で学習済みの検出ヘッドは80クラス用であり，そのままではグーチョキパー用に使えない。ここでは MobileNetV3 の ImageNet 事前学習済み重みを backbone に使い，検出ヘッドはグーチョキパー用に新しく作る。

この考え方が「MobileNetV3 を特徴抽出機として使って，SSD をファインチューニングする」ということである。


<div class="od-figure">
<svg viewBox="0 0 900 260" role="img" aria-label="ファインチューニングの考え方">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="35" y="55" width="190" height="130" class="od-box od-green"/><text x="130" y="102" text-anchor="middle" class="od-label">ImageNet pretrained</text><text x="130" y="130" text-anchor="middle" class="od-small">MobileNetV3 backbone</text>
<path d="M240 120 H330" class="od-arrow"/>
<rect x="345" y="55" width="180" height="130" class="od-box od-amber"/><text x="435" y="103" text-anchor="middle" class="od-label">新しいSSD head</text><text x="435" y="130" text-anchor="middle" class="od-small">背景 + gu/choki/pa</text>
<path d="M540 120 H630" class="od-arrow"/>
<rect x="645" y="55" width="205" height="130" class="od-box od-red"/><text x="747" y="102" text-anchor="middle" class="od-label">RPS dataset</text><text x="747" y="130" text-anchor="middle" class="od-small">手の位置とラベルで更新</text>
</svg>

<div class="od-note">COCO用ヘッドではなく，グーチョキパー用のクラス数に合わせた検出ヘッドを学習する。</div>
</div>


In [ ]:
RPS_CLASSES = ["background", "gu", "choki", "pa"]
LABEL_TO_ID = {label: index for index, label in enumerate(RPS_CLASSES)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}


def make_rps_detector(num_classes: int = len(RPS_CLASSES)) -> nn.Module:
    """グーチョキパー検出用の SSD + MobileNetV3 を作る."""

    model = ssdlite320_mobilenet_v3_large(
        weights=None,
        weights_backbone=MobileNet_V3_Large_Weights.DEFAULT,
        num_classes=num_classes,
    )
    return model


rps_detector = make_rps_detector().to(DEVICE)
print(f"classes including background: {len(RPS_CLASSES)}")


## カメラ動画を教師データにする考え方

カメラで動画を撮影すると，似た画像を短時間でたくさん集められる。ただし，動画の全フレームを使うとほぼ同じ画像ばかりになり，データが偏りやすい。

実用的には，次の流れにする。

1. `gu`，`choki`，`pa` をそれぞれ複数の背景，距離，角度，明るさで撮影する。
2. 動画から数フレームおきに静止画を切り出す。
3. 各画像で手を囲むバウンディングボックスとラベルを付ける。
4. 訓練データ，検証データ，テストデータに分ける。
5. 訓練時だけ，色や位置を少し変えるデータ拡張を行う。

注意点として，同じ動画から連続して切り出した画像を train と test に混ぜると，ほぼ同じ画像で評価することになる。できれば動画単位，撮影条件単位，または撮影日単位で分割する。


<div class="od-figure">
<svg viewBox="0 0 900 260" role="img" aria-label="動画から教師データを作る流れ">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="70" width="120" height="90" class="od-box od-blue"/><text x="100" y="108" text-anchor="middle" class="od-label">カメラ</text><text x="100" y="132" text-anchor="middle" class="od-small">動画</text>
<path d="M175 115 H245" class="od-arrow"/>
<rect x="260" y="70" width="120" height="90" class="od-box od-green"/><text x="320" y="108" text-anchor="middle" class="od-label">フレーム</text><text x="320" y="132" text-anchor="middle" class="od-small">間引き</text>
<path d="M395 115 H465" class="od-arrow"/>
<rect x="480" y="70" width="140" height="90" class="od-box od-amber"/><text x="550" y="108" text-anchor="middle" class="od-label">アノテーション</text><text x="550" y="132" text-anchor="middle" class="od-small">box + label</text>
<path d="M635 115 H705" class="od-arrow"/>
<rect x="720" y="70" width="140" height="90" class="od-box od-red"/><text x="790" y="108" text-anchor="middle" class="od-label">学習データ</text><text x="790" y="132" text-anchor="middle" class="od-small">train / val</text>
</svg>

<div class="od-note">動画をそのまま使うのではなく，静止画とアノテーションへ変換してから学習に使う。</div>
</div>


## ノートブック上で動画を撮影する

次のセルは，ブラウザのカメラを使って短い動画を録画し，`.webm` ファイルとしてダウンロードするための簡易ツールである。ダウンロードされた動画を `../data/rps_detection/videos/` に置く。

Jupyter を開いているブラウザがカメラ利用を許可していない場合は，PC やスマートフォンのカメラアプリで撮影した動画を同じフォルダへ置けばよい。


<div class="od-figure">
<svg viewBox="0 0 760 240" role="img" aria-label="録画セルの役割">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="50" y="45" width="250" height="150" class="od-box od-blue"/><rect x="80" y="75" width="190" height="90" fill="#dbeafe" stroke="#0284c7"/><circle cx="175" cy="120" r="28" fill="#93c5fd"/><text x="175" y="220" text-anchor="middle" class="od-label">ブラウザのカメラ</text>
<path d="M330 120 H435" class="od-arrow"/>
<rect x="460" y="65" width="220" height="110" class="od-box od-green"/><text x="570" y="107" text-anchor="middle" class="od-label">rps-*.webm</text><text x="570" y="133" text-anchor="middle" class="od-small">videos フォルダへ配置</text>
</svg>

<div class="od-note">録画セルはデータ収集の入口であり，最終的には動画ファイルをデータフォルダへ置く。</div>
</div>


In [ ]:
from IPython.display import HTML

HTML("""
<div style="display:flex; gap:12px; align-items:flex-start; flex-wrap:wrap;">
  <video id="rps-preview" autoplay muted playsinline style="width:320px; border:1px solid #ccc;"></video>
  <div style="display:flex; flex-direction:column; gap:8px;">
    <button id="rps-start">Start recording</button>
    <button id="rps-stop" disabled>Stop and download</button>
    <span id="rps-status">idle</span>
  </div>
</div>
<script>
(async () => {
  const video = document.getElementById('rps-preview');
  const startButton = document.getElementById('rps-start');
  const stopButton = document.getElementById('rps-stop');
  const status = document.getElementById('rps-status');
  const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
  video.srcObject = stream;
  let recorder;
  let chunks = [];
  startButton.onclick = () => {
    chunks = [];
    recorder = new MediaRecorder(stream, {mimeType: 'video/webm'});
    recorder.ondataavailable = event => chunks.push(event.data);
    recorder.onstop = () => {
      const blob = new Blob(chunks, {type: 'video/webm'});
      const url = URL.createObjectURL(blob);
      const a = document.createElement('a');
      a.href = url;
      a.download = `rps-${Date.now()}.webm`;
      a.click();
      URL.revokeObjectURL(url);
      status.textContent = 'downloaded';
    };
    recorder.start();
    startButton.disabled = true;
    stopButton.disabled = false;
    status.textContent = 'recording';
  };
  stopButton.onclick = () => {
    recorder.stop();
    startButton.disabled = false;
    stopButton.disabled = true;
  };
})();
</script>
""")


## 動画からフレームを切り出す

`../data/rps_detection/videos/` に置いた動画から，数フレームおきに画像を保存する。似たフレームばかりを増やさないため，`every_n_frames` を大きくすると間引きが強くなる。

動画読み込みには環境によって追加ライブラリが必要な場合がある。うまく読めない場合は，動画プレイヤーや `ffmpeg` などで静止画へ変換して，`../data/rps_detection/images/` に置いてもよい。


<div class="od-figure">
<svg viewBox="0 0 850 250" role="img" aria-label="動画からフレーム抽出">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="80" width="230" height="70" class="od-box od-blue"/><g class="od-scanbar"><rect x="55" y="85" width="22" height="60" fill="#0284c7" opacity="0.8"/></g><text x="155" y="175" text-anchor="middle" class="od-label">動画タイムライン</text>
<path d="M290 115 H385" class="od-arrow"/>
<rect x="405" y="55" width="80" height="80" class="od-box od-green"/><rect x="505" y="55" width="80" height="80" class="od-box od-green"/><rect x="605" y="55" width="80" height="80" class="od-box od-green"/><text x="545" y="175" text-anchor="middle" class="od-label">一定間隔で保存</text>
</svg>

<div class="od-note">連続フレームをすべて使うより，間引いて多様性を確保する。</div>
</div>


In [ ]:
def extract_frames_from_video(
    video_path: Path,
    output_dir: Path,
    every_n_frames: int = 10,
    max_frames: int | None = None,
) -> list[Path]:
    """動画から一定間隔でフレームを切り出して保存する."""

    output_dir.mkdir(parents=True, exist_ok=True)
    try:
        from torchvision.io import read_video
    except ImportError as error:
        message = (
            "この環境の torchvision では read_video を利用できません。"
            "動画プレイヤーや ffmpeg で静止画へ変換し，"
            f"{output_dir} に保存してください。"
        )
        raise RuntimeError(message) from error

    video, _, _ = read_video(str(video_path), pts_unit="sec", output_format="TCHW")

    saved_paths: list[Path] = []
    for frame_index in range(0, len(video), every_n_frames):
        if max_frames is not None and len(saved_paths) >= max_frames:
            break
        frame = video[frame_index].permute(1, 2, 0).numpy()
        image = Image.fromarray(frame)
        image_path = output_dir / f"{video_path.stem}_{frame_index:06d}.jpg"
        image.save(image_path, quality=95)
        saved_paths.append(image_path)
    return saved_paths


video_dir = RPS_ROOT / "videos"
image_dir = RPS_ROOT / "images"
video_paths = sorted(video_dir.glob("*.mp4")) + sorted(video_dir.glob("*.webm"))

if video_paths:
    saved = extract_frames_from_video(video_paths[0], image_dir, every_n_frames=10, max_frames=30)
    print(f"saved {len(saved)} frames from {video_paths[0]}")
else:
    print(f"動画を置くフォルダ: {video_dir}")


## アノテーションCSVの形式

物体検出の教師データには，画像ごとにバウンディングボックスとラベルが必要である。このノートでは，次のCSV形式を使う。

| image | xmin | ymin | xmax | ymax | label |
|---|---:|---:|---:|---:|---|
| gu_0001.jpg | 120 | 80 | 260 | 240 | gu |
| choki_0001.jpg | 90 | 60 | 250 | 260 | choki |
| pa_0001.jpg | 110 | 70 | 280 | 270 | pa |

手が1つだけ写っているなら，1画像につき1行でよい。1枚の画像に複数の手が写る場合は，同じ `image` に対して複数行を書く。

ラベル付けは，Label Studio，CVAT，Roboflow，VGG Image Annotator などを使うと効率がよい。重要なのは，画像分類と違って「画像名とラベル」だけでは足りず，「どこにあるか」の座標も必要になる点である。


<div class="od-figure">
<svg viewBox="0 0 900 250" role="img" aria-label="アノテーションCSVの構造">
<rect x="35" y="35" width="830" height="170" class="od-box od-blue"/>
<line x1="35" y1="80" x2="865" y2="80" stroke="#486581"/>
<line x1="190" y1="35" x2="190" y2="205" stroke="#486581"/><line x1="310" y1="35" x2="310" y2="205" stroke="#486581"/><line x1="430" y1="35" x2="430" y2="205" stroke="#486581"/><line x1="550" y1="35" x2="550" y2="205" stroke="#486581"/><line x1="670" y1="35" x2="670" y2="205" stroke="#486581"/>
<text x="112" y="64" text-anchor="middle" class="od-label">image</text><text x="250" y="64" text-anchor="middle" class="od-label">xmin</text><text x="370" y="64" text-anchor="middle" class="od-label">ymin</text><text x="490" y="64" text-anchor="middle" class="od-label">xmax</text><text x="610" y="64" text-anchor="middle" class="od-label">ymax</text><text x="767" y="64" text-anchor="middle" class="od-label">label</text>
<text x="112" y="122" text-anchor="middle" class="od-small">gu_0001.jpg</text><text x="250" y="122" text-anchor="middle" class="od-small">120</text><text x="370" y="122" text-anchor="middle" class="od-small">80</text><text x="490" y="122" text-anchor="middle" class="od-small">260</text><text x="610" y="122" text-anchor="middle" class="od-small">240</text><text x="767" y="122" text-anchor="middle" class="od-small">gu</text>
<text x="112" y="165" text-anchor="middle" class="od-small">pa_0001.jpg</text><text x="250" y="165" text-anchor="middle" class="od-small">110</text><text x="370" y="165" text-anchor="middle" class="od-small">70</text><text x="490" y="165" text-anchor="middle" class="od-small">280</text><text x="610" y="165" text-anchor="middle" class="od-small">270</text><text x="767" y="165" text-anchor="middle" class="od-small">pa</text>
</svg>

<div class="od-note">1行が1つの物体に対応する。同じ画像に複数物体がある場合は，同じ画像名の行を増やす。</div>
</div>


In [ ]:
annotation_csv = RPS_ROOT / "annotations.csv"

example_annotations = pd.DataFrame(
    [
        {"image": "gu_0001.jpg", "xmin": 120, "ymin": 80, "xmax": 260, "ymax": 240, "label": "gu"},
        {"image": "choki_0001.jpg", "xmin": 90, "ymin": 60, "xmax": 250, "ymax": 260, "label": "choki"},
        {"image": "pa_0001.jpg", "xmin": 110, "ymin": 70, "xmax": 280, "ymax": 270, "label": "pa"},
    ]
)

example_annotations


## Dataset を作る

TorchVision の検出モデルは，1枚の画像に対応する教師データを辞書で受け取る。最低限必要なのは，次の2つである。

- `boxes`：`[num_objects, 4]` の Tensor。座標は `XYXY` 形式。
- `labels`：`[num_objects]` の Tensor。背景以外のクラスIDを入れる。

ここでは評価や可視化にも使いやすいように，`image_id`，`area`，`iscrowd` も入れておく。


<div class="od-figure">
<svg viewBox="0 0 850 250" role="img" aria-label="Dataset が返す形式">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="45" y="70" width="150" height="100" class="od-box od-blue"/><text x="120" y="112" text-anchor="middle" class="od-label">画像ファイル</text><text x="120" y="138" text-anchor="middle" class="od-small">PIL image</text>
<rect x="45" y="185" width="150" height="40" class="od-box od-amber"/><text x="120" y="211" text-anchor="middle" class="od-small">CSV rows</text>
<path d="M215 120 H310" class="od-arrow"/>
<rect x="325" y="55" width="180" height="130" class="od-box od-green"/><text x="415" y="100" text-anchor="middle" class="od-label">Dataset</text><text x="415" y="126" text-anchor="middle" class="od-small">__getitem__</text>
<path d="M520 120 H615" class="od-arrow"/>
<rect x="630" y="45" width="180" height="150" class="od-box od-red"/><text x="720" y="84" text-anchor="middle" class="od-label">image tensor</text><text x="720" y="118" text-anchor="middle" class="od-label">target dict</text><text x="720" y="148" text-anchor="middle" class="od-small">boxes, labels, area</text>
</svg>

<div class="od-note">DataLoader に渡す前に，画像と教師データをモデルが期待する形式へ変換する。</div>
</div>


In [ ]:
class RpsDetectionDataset(Dataset[tuple[torch.Tensor, dict[str, torch.Tensor]]]):
    """グーチョキパー物体検出用 Dataset."""

    def __init__(self, image_dir: Path, annotation_csv: Path, transform: transforms.Compose | None = None) -> None:
        self.image_dir = image_dir
        self.transform = transform
        annotations = pd.read_csv(annotation_csv)
        self.image_names = sorted(annotations["image"].unique().tolist())
        self.records = {name: group.reset_index(drop=True) for name, group in annotations.groupby("image")}

    def __len__(self) -> int:
        return len(self.image_names)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        image_name = self.image_names[index]
        image = Image.open(self.image_dir / image_name).convert("RGB")
        width, height = image.size
        rows = self.records[image_name]

        boxes = torch.as_tensor(rows[["xmin", "ymin", "xmax", "ymax"]].to_numpy(), dtype=torch.float32)
        labels = torch.as_tensor([LABEL_TO_ID[label] for label in rows["label"]], dtype=torch.int64)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

        image_tv = tv_tensors.Image(image)
        target: dict[str, torch.Tensor] = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(height, width)),
            "labels": labels,
            "image_id": torch.tensor([index], dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(rows),), dtype=torch.int64),
        }

        if self.transform is not None:
            image_tv, target = self.transform(image_tv, target)
        return image_tv, target


## データ拡張

データ拡張は，訓練画像を少し変えて，モデルが撮影条件の違いに強くなるようにする方法である。グーチョキパー検出では，手の位置，明るさ，色味，カメラとの距離，背景が変わりやすい。

物体検出では，画像だけを反転・リサイズしてはいけない。バウンディングボックスの座標も同じ変換に合わせて更新する必要がある。ここでは TorchVision v2 transforms を使い，画像と `BoundingBoxes` を同時に変換する。


<div class="od-figure">
<svg viewBox="0 0 860 280" role="img" aria-label="バウンディングボックス対応データ拡張">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="45" y="45" width="220" height="160" class="od-box od-blue"/><rect x="105" y="85" width="80" height="90" fill="none" stroke="#dc2626" stroke-width="4"/><text x="155" y="235" text-anchor="middle" class="od-label">元画像</text>
<path d="M285 125 H390" class="od-arrow"/>
<rect x="410" y="45" width="220" height="160" class="od-box od-green"/><g class="od-shift"><rect x="500" y="75" width="80" height="90" fill="none" stroke="#dc2626" stroke-width="4"/></g><text x="520" y="235" text-anchor="middle" class="od-label">画像とboxを同時変換</text>
<path d="M650 125 H755" class="od-arrow"/>
<rect x="775" y="65" width="55" height="120" class="od-box od-amber"/><text x="802" y="220" text-anchor="middle" class="od-label">学習へ</text>
</svg>

<div class="od-note">検出では画像だけを変換すると教師ボックスがずれるため，画像と BoundingBoxes を同時に変換する。</div>
</div>


In [ ]:
def make_detection_transform(train: bool) -> transforms.Compose:
    """物体検出用の前処理とデータ拡張を作る."""

    transform_list: list[nn.Module] = []
    if train:
        transform_list.extend(
            [
                transforms.RandomPhotometricDistort(p=0.5),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomZoomOut(fill=0, p=0.3),
            ]
        )

    transform_list.extend(
        [
            transforms.Resize((320, 320)),
            transforms.ToDtype(torch.float32, scale=True),
            transforms.ToPureTensor(),
        ]
    )
    return transforms.Compose(transform_list)


def collate_detection_batch(
    batch: list[tuple[torch.Tensor, dict[str, torch.Tensor]]],
) -> tuple[list[torch.Tensor], list[dict[str, torch.Tensor]]]:
    """物体検出用に，画像と教師データをリストのまままとめる."""

    images, targets = zip(*batch, strict=True)
    return list(images), list(targets)


## Dataset と DataLoader を用意する

まだアノテーションCSVを作っていない場合，次のセルは学習をスキップする。データを用意した後に再実行すれば，DataLoader が作られる。


<div class="od-figure">
<svg viewBox="0 0 860 240" role="img" aria-label="DataLoader と collate 関数">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="60" width="160" height="100" class="od-box od-blue"/><text x="120" y="100" text-anchor="middle" class="od-label">Dataset</text><text x="120" y="126" text-anchor="middle" class="od-small">1枚ずつ返す</text>
<path d="M215 110 H310" class="od-arrow"/>
<rect x="325" y="60" width="170" height="100" class="od-box od-green"/><text x="410" y="100" text-anchor="middle" class="od-label">collate_fn</text><text x="410" y="126" text-anchor="middle" class="od-small">リストのまま束ねる</text>
<path d="M510 110 H605" class="od-arrow"/>
<rect x="620" y="45" width="200" height="130" class="od-box od-red"/><text x="720" y="88" text-anchor="middle" class="od-label">images: list</text><text x="720" y="118" text-anchor="middle" class="od-label">targets: list</text><text x="720" y="148" text-anchor="middle" class="od-small">物体数が画像ごとに違う</text>
</svg>

<div class="od-note">検出では画像ごとに物体数が違うため，通常のTensor積み上げではなくリストとして渡す。</div>
</div>


In [ ]:
train_loader: DataLoader | None = None
val_loader: DataLoader | None = None

if annotation_csv.exists():
    full_dataset_for_split = RpsDetectionDataset(image_dir, annotation_csv, transform=None)
    indices = np.random.default_rng(SEED).permutation(len(full_dataset_for_split))
    split = max(1, int(len(indices) * 0.8))
    train_indices = indices[:split].tolist()
    val_indices = indices[split:].tolist()

    train_dataset = Subset(
        RpsDetectionDataset(image_dir, annotation_csv, transform=make_detection_transform(train=True)),
        train_indices,
    )
    val_dataset = Subset(
        RpsDetectionDataset(image_dir, annotation_csv, transform=make_detection_transform(train=False)),
        val_indices,
    )

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0, collate_fn=collate_detection_batch)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0, collate_fn=collate_detection_batch)
    print(f"train images: {len(train_dataset)}, val images: {len(val_dataset)}")
else:
    print(f"アノテーションCSVを作成してください: {annotation_csv}")


## 教師データを確認する

学習前に，バウンディングボックスが画像上の正しい位置にあるかを必ず確認する。ラベルや座標がずれていると，モデルは正しい検出を学習できない。


<div class="od-figure">
<svg viewBox="0 0 760 250" role="img" aria-label="教師データ確認">
<rect x="55" y="40" width="260" height="170" class="od-box od-blue"/><rect x="118" y="72" width="105" height="115" fill="none" stroke="#16a34a" stroke-width="5"/><text x="170" y="65" text-anchor="middle" fill="#166534" font-size="14" font-family="system-ui">gu</text>
<rect x="420" y="65" width="280" height="120" class="od-box od-amber"/><text x="560" y="105" text-anchor="middle" class="od-label">学習前に確認</text><text x="560" y="134" text-anchor="middle" class="od-small">ラベル・座標のずれを見つける</text>
<path d="M345 125 H420" class="od-arrow"/>
</svg>

<div class="od-note">検出モデルの失敗原因は，モデルよりアノテーションのずれであることも多い。</div>
</div>


In [ ]:
def draw_target(image_tensor: torch.Tensor, target: dict[str, torch.Tensor]) -> Image.Image:
    """Dataset の教師ボックスを画像へ描画する."""

    image_array = (image_tensor.detach().permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    canvas = Image.fromarray(image_array)
    draw = ImageDraw.Draw(canvas)

    for box, label_id in zip(target["boxes"].detach().cpu(), target["labels"].detach().cpu(), strict=True):
        xyxy = [float(value) for value in box]
        draw.rectangle(xyxy, outline="lime", width=3)
        draw.text((xyxy[0], max(0, xyxy[1] - 14)), ID_TO_LABEL[int(label_id)], fill="lime")
    return canvas


if train_loader is not None:
    images_batch, targets_batch = next(iter(train_loader))
    draw_target(images_batch[0], targets_batch[0])
else:
    print("データセット作成後に実行してください。")


## 訓練ループ

TorchVision の物体検出モデルは，訓練モードで `model(images, targets)` を呼び出すと，複数の損失を辞書で返す。SSD では主に，分類の損失とボックス回帰の損失を最小化する。

CPU では時間がかかるため，まずは少ないエポック数で動作確認する。データが増えたら，エポック数や学習率を調整する。


<div class="od-figure">
<svg viewBox="0 0 880 250" role="img" aria-label="訓練ループ">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="70" width="130" height="90" class="od-box od-blue"/><text x="105" y="108" text-anchor="middle" class="od-label">batch</text><text x="105" y="132" text-anchor="middle" class="od-small">images + targets</text>
<path d="M185 115 H260" class="od-arrow"/>
<rect x="275" y="70" width="130" height="90" class="od-box od-green"/><text x="340" y="108" text-anchor="middle" class="od-label">model</text><text x="340" y="132" text-anchor="middle" class="od-small">loss dict</text>
<path d="M420 115 H495" class="od-arrow"/>
<rect x="510" y="70" width="130" height="90" class="od-box od-amber"/><text x="575" y="108" text-anchor="middle" class="od-label">backward</text><text x="575" y="132" text-anchor="middle" class="od-small">勾配</text>
<path d="M655 115 H730" class="od-arrow"/>
<rect x="745" y="70" width="100" height="90" class="od-box od-red"/><text x="795" y="108" text-anchor="middle" class="od-label">update</text><text x="795" y="132" text-anchor="middle" class="od-small">SGD</text>
</svg>

<div class="od-note">分類損失とボックス回帰損失を合計し，通常のPyTorchと同じように逆伝播する。</div>
</div>


In [ ]:
def move_targets_to_device(
    targets: list[dict[str, torch.Tensor]],
    device: torch.device,
) -> list[dict[str, torch.Tensor]]:
    """教師データ内の Tensor を指定デバイスへ移す."""

    return [{key: value.to(device) for key, value in target.items()} for target in targets]


def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> dict[str, float]:
    """物体検出モデルを1エポック訓練する."""

    model.train()
    loss_sums: defaultdict[str, float] = defaultdict(float)
    num_batches = 0

    for images, targets in data_loader:
        images = [image.to(device) for image in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_value for loss_value in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for key, value in loss_dict.items():
            loss_sums[key] += float(value.detach().cpu())
        loss_sums["total"] += float(loss.detach().cpu())
        num_batches += 1

    return {key: value / max(1, num_batches) for key, value in loss_sums.items()}


In [ ]:
NUM_EPOCHS = 2
LEARNING_RATE = 1e-3

if train_loader is not None:
    rps_detector = make_rps_detector().to(DEVICE)
    optimizer = torch.optim.SGD(
        [parameter for parameter in rps_detector.parameters() if parameter.requires_grad],
        lr=LEARNING_RATE,
        momentum=0.9,
        weight_decay=5e-4,
    )

    for epoch in range(NUM_EPOCHS):
        losses = train_one_epoch(rps_detector, train_loader, optimizer, DEVICE)
        loss_text = ", ".join(f"{key}: {value:.4f}" for key, value in losses.items())
        print(f"epoch {epoch + 1}: {loss_text}")
else:
    print("データセット作成後に実行してください。")


## ファインチューニング後の推論

ファインチューニング後は，モデルを評価モードにして画像だけを渡す。返ってくる形式は COCO 事前学習済みモデルと同じで，`boxes`，`labels`，`scores` が含まれる。


<div class="od-figure">
<svg viewBox="0 0 820 250" role="img" aria-label="ファインチューニング後の推論">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="45" y="65" width="170" height="110" class="od-box od-blue"/><text x="130" y="108" text-anchor="middle" class="od-label">新しい画像</text><text x="130" y="134" text-anchor="middle" class="od-small">手を写す</text>
<path d="M230 120 H320" class="od-arrow"/>
<rect x="335" y="65" width="170" height="110" class="od-box od-green"/><text x="420" y="108" text-anchor="middle" class="od-label">fine-tuned SSD</text><text x="420" y="134" text-anchor="middle" class="od-small">eval</text>
<path d="M520 120 H610" class="od-arrow"/>
<rect x="625" y="50" width="150" height="140" class="od-box od-red"/><text x="700" y="92" text-anchor="middle" class="od-label">gu/choki/pa</text><text x="700" y="122" text-anchor="middle" class="od-small">box</text><text x="700" y="148" text-anchor="middle" class="od-small">score</text>
</svg>

<div class="od-note">学習後の出力形式はCOCOモデルと同じで，カテゴリ名だけがグーチョキパー用になる。</div>
</div>


In [ ]:
def predict_rps(
    model: nn.Module,
    image: Image.Image,
    score_threshold: float = 0.5,
) -> Image.Image:
    """グーチョキパー検出結果を描画する."""

    model.eval()
    transform = make_detection_transform(train=False)
    image_tensor = transform(tv_tensors.Image(image)).to(DEVICE)

    with torch.inference_mode():
        prediction = model([image_tensor])[0]

    prediction = {key: value.cpu() for key, value in prediction.items()}
    categories = [ID_TO_LABEL[index] for index in range(len(ID_TO_LABEL))]
    return draw_torchvision_detections(image, prediction, categories, score_threshold=score_threshold)


if val_loader is not None:
    val_subset = val_loader.dataset
    image_tensor, _ = val_subset[0]
    pil_image = Image.fromarray((image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
    predict_rps(rps_detector, pil_image, score_threshold=0.3)
else:
    print("データセット作成後に実行してください。")


## 学習を安定させるための注意点

グーチョキパー検出では，モデルやコードよりもデータの作り方が性能に大きく影響する。

- 各クラスの枚数をなるべく揃える。
- 背景，明るさ，手の大きさ，向きを変えて撮影する。
- 同じ動画の連続フレームだけでデータを増やしすぎない。
- バウンディングボックスは手全体を少し余裕をもって囲む。
- train と test にほぼ同じフレームが入らないように分割する。
- データ拡張は訓練データだけに使い，検証・テストデータには使わない。

検出モデルでは「分類が当たっているか」だけでなく，「ボックスの位置がどれだけ正しいか」も重要である。正式な評価には mAP を使うが，まずは予測画像を可視化して，どの条件で失敗するかを観察する。


<div class="od-figure">
<svg viewBox="0 0 900 270" role="img" aria-label="データ品質の観点">
<rect x="45" y="55" width="150" height="100" class="od-box od-blue"/><text x="120" y="95" text-anchor="middle" class="od-label">枚数</text><text x="120" y="120" text-anchor="middle" class="od-small">クラス均衡</text>
<rect x="225" y="55" width="150" height="100" class="od-box od-green"/><text x="300" y="95" text-anchor="middle" class="od-label">多様性</text><text x="300" y="120" text-anchor="middle" class="od-small">背景・明るさ</text>
<rect x="405" y="55" width="150" height="100" class="od-box od-amber"/><text x="480" y="95" text-anchor="middle" class="od-label">分割</text><text x="480" y="120" text-anchor="middle" class="od-small">動画単位</text>
<rect x="585" y="55" width="150" height="100" class="od-box od-red"/><text x="660" y="95" text-anchor="middle" class="od-label">box品質</text><text x="660" y="120" text-anchor="middle" class="od-small">手全体を囲む</text>
<rect x="765" y="55" width="90" height="100" class="od-box od-purple"/><text x="810" y="95" text-anchor="middle" class="od-label">評価</text><text x="810" y="120" text-anchor="middle" class="od-small">可視化</text>
<path d="M120 180 C220 230 700 230 810 180" class="od-dash"/>
<text x="465" y="235" text-anchor="middle" class="od-label">性能はデータ設計に強く依存する</text>
</svg>

<div class="od-note">少量データでは，モデル選択よりも撮影・分割・ラベル付けの設計が効きやすい。</div>
</div>


## まとめ

このノートブックでは，SSD + MobileNetV3 による物体検出を扱った。

- 画像分類は画像全体のラベルを出すが，物体検出は物体ごとのラベルと位置を出す。
- MobileNetV3 は，画像から特徴マップを作る特徴抽出機として使える。
- SSD は，多数の既定ボックスに対してクラスと位置のずれを予測する。
- COCO 事前学習済みモデルを使うと，追加学習なしで一般物体を検出できる。
- グーチョキパーのような独自クラスでは，動画からフレームを切り出し，ボックス付き教師データを作ってファインチューニングする。
- データ拡張では，画像だけでなくバウンディングボックスも同時に変換する必要がある。

物体検出では，モデル構造の理解と同じくらい，教師データの品質が重要である。撮影条件を変え，正確なアノテーションを付けることが，安定した検出につながる。


<div class="od-figure">
<svg viewBox="0 0 900 260" role="img" aria-label="まとめの全体像">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#486581" /></marker></defs>
<rect x="40" y="70" width="130" height="90" class="od-box od-blue"/><text x="105" y="108" text-anchor="middle" class="od-label">画像</text>
<path d="M185 115 H255" class="od-arrow"/><rect x="270" y="70" width="155" height="90" class="od-box od-green"/><text x="347" y="103" text-anchor="middle" class="od-label">特徴抽出</text><text x="347" y="128" text-anchor="middle" class="od-small">MobileNetV3</text>
<path d="M440 115 H510" class="od-arrow"/><rect x="525" y="70" width="125" height="90" class="od-box od-amber"/><text x="587" y="108" text-anchor="middle" class="od-label">SSD</text><text x="587" y="132" text-anchor="middle" class="od-small">box + class</text>
<path d="M665 115 H735" class="od-arrow"/><rect x="750" y="70" width="110" height="90" class="od-box od-red"/><text x="805" y="108" text-anchor="middle" class="od-label">検出</text><text x="805" y="132" text-anchor="middle" class="od-small">RPSへ応用</text>
</svg>

<div class="od-note">物体検出は，特徴抽出・候補ボックス・分類と位置補正・データ作成をつなげて考える。</div>
</div>
